In [ ]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.optim as Optimizers
import torch.nn as nn
import torchvision
import pandas as pd
import numpy
from sklearn.preprocessing import StandardScaler

In [2]:
data_frame = pd.read_csv('diabetes.csv')
data_frame


,Number of times pregnant,Plasma glucose concentration,Diastolic blood pressure,Triceps skin fold thickness,2-Hour serum insulin,Body mass index,Age,Class
0,6,148,72,35,0,33.6,50,positive
1,1,85,66,29,0,26.6,31,negative
2,8,183,64,0,0,23.3,32,positive
3,1,89,66,23,94,28.1,21,negative
4,0,137,40,35,168,43.1,33,positive
...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,63,negative
764,2,122,70,27,0,36.8,27,negative
765,5,121,72,23,112,26.2,30,negative
766,1,126,60,0,0,30.1,47,positive


In [3]:
x = data_frame.iloc[: -50, :-1].values
x_test = data_frame.iloc[718:, :-1].values
print(x.shape)
x
print(x_test.shape)

(718, 7)
(50, 7)


In [4]:
y = data_frame.iloc[: -50, -1].values
y_test = data_frame.iloc[718:, -1].values
y = (y == 'positive').astype(float)
y_test = (y_test == 'positive').astype(float)
print(y.shape)
print(y_test.shape)
y

(718,)
(50,)


array([1., 0., 1., 0., 1., 0., 1., 0., 1., 1., 0., 1., 0., 1., 1., 1., 1.,
       1., 0., 1., 0., 0., 1., 1., 1., 1., 1., 0., 0., 0., 0., 1., 0., 0.,
       0., 0., 0., 1., 1., 1., 0., 0., 0., 1., 0., 1., 0., 0., 1., 0., 0.,
       0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 1., 0., 1., 0.,
       0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1.,
       0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 1., 0.,
       0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 0., 0., 1., 1., 1., 0., 0.,
       0., 1., 0., 0., 0., 1., 1., 0., 0., 1., 1., 1., 1., 1., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
       0., 1., 1., 0., 0., 0., 1., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0.,
       1., 1., 0., 0., 0., 1., 0., 1., 0., 1., 0., 0., 0., 0., 0., 1., 1.,
       1., 1., 1., 0., 0., 1., 1., 0., 1., 0., 1., 1., 1., 0., 0., 0., 0.,
       0., 0., 1., 1., 0., 1., 0., 0., 0., 1., 1., 1., 1., 0., 1., 1., 1.,
       1., 0., 0., 0., 0.

In [5]:
# Normalize data
scaler = StandardScaler()
x = scaler.fit_transform(x)
x_test = scaler.transform(x_test)
x

array([[ 0.63977559,  0.84704983,  0.16677275, ..., -0.69365831,
         0.21698733,  1.43398388],
       [-0.84778553, -1.11292009, -0.1379164 , ..., -0.69365831,
        -0.66123649, -0.18354614],
       [ 1.23480003,  1.935922  , -0.23947945, ..., -0.69365831,
        -1.07525629, -0.09841298],
       ...,
       [ 0.93728781,  2.06036454, -0.9504208 , ...,  2.67314676,
         0.2546255 ,  0.07185333],
       [-0.25276108,  1.62481567,  0.4714619 , ...,  0.89526755,
         0.24207944, -0.18354614],
       [ 1.82982448, -0.83292439,  0.16677275, ..., -0.69365831,
        -1.1003484 ,  1.94478283]])

In [6]:
x = torch.tensor(x, requires_grad=True)
x_test = torch.tensor(x_test)
print(x_test.shape)
x.shape

torch.Size([50, 7])


torch.Size([718, 7])

In [7]:
y = torch.tensor(y, requires_grad=True)
y_test = torch.tensor(y_test)
y = y.detach().clone().requires_grad_(True).unsqueeze(-1)
y_test = y_test.detach().clone().requires_grad_(True).unsqueeze(-1)
print(y_test.shape)
y.shape

torch.Size([50, 1])


torch.Size([718, 1])

In [8]:
class CustomDataSet(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y
    
    def __getitem__(self, index):
        return self.x[index], self.y[index]
    
    def __len__(self):
        return len(self.x)


In [ ]:
data_set = CustomDataSet(x, y)
test_data = CustomDataSet(x_test, y_test)
train_dataloader = DataLoader(dataset=data_set, batch_size=64, shuffle=True)
test_dataloader = DataLoader(dataset=test_data)

In [33]:
class diabetesModel(nn.Module):
    def __init__(self, input_size, output_size):
        super(diabetesModel, self).__init__()
        self.input_layer = nn.Linear(input_size, 5)
        self.hidden_layer1 = nn.Linear(5, 4)
        self.hidden_layer2 = nn.Linear(4, 3)
        self.output_layer = nn.Linear(3, output_size)
        self.hidden_activate_func = nn.ReLU()
        self.output_activate_func = nn.Sigmoid()
        
    def forward(self, train_data):
        train_data = self.input_layer(train_data)
        train_data = self.hidden_activate_func(train_data)
        train_data = self.hidden_layer1(train_data)
        train_data = self.hidden_activate_func(train_data)
        train_data = self.hidden_layer2(train_data)
        train_data = self.hidden_activate_func(train_data)
        train_data = self.output_layer(train_data)
        train_data = self.output_activate_func(train_data)
        return train_data

In [58]:
model = diabetesModel(7, 1)
criterion = nn.BCELoss()
optimizer = Optimizers.AdamW(model.parameters()) 

In [59]:
num_epochs = 200
for epoch in range(0, num_epochs):
    i = 0
    for inputs, labels in train_dataloader:
        inputs = inputs.float()
        labels = labels.float()
        output = model(inputs)
        loss = criterion(output, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    output = (output>0.5).float()
    accuracy = (output == labels).float().mean()
    print("Epoch {}/{}, Loss: {:.3f}, Accuracy: {:.3f}".format(epoch+1,num_epochs, loss, accuracy))
        

Epoch 1/200, Loss: 0.738, Accuracy: 0.429
Epoch 2/200, Loss: 0.733, Accuracy: 0.429
Epoch 3/200, Loss: 0.730, Accuracy: 0.429
Epoch 4/200, Loss: 0.728, Accuracy: 0.429
Epoch 5/200, Loss: 0.726, Accuracy: 0.429
Epoch 6/200, Loss: 0.723, Accuracy: 0.429
Epoch 7/200, Loss: 0.721, Accuracy: 0.429
Epoch 8/200, Loss: 0.717, Accuracy: 0.429
Epoch 9/200, Loss: 0.711, Accuracy: 0.429
Epoch 10/200, Loss: 0.705, Accuracy: 0.429
Epoch 11/200, Loss: 0.701, Accuracy: 0.429
Epoch 12/200, Loss: 0.696, Accuracy: 0.429
Epoch 13/200, Loss: 0.691, Accuracy: 0.429
Epoch 14/200, Loss: 0.687, Accuracy: 0.500
Epoch 15/200, Loss: 0.682, Accuracy: 0.857
Epoch 16/200, Loss: 0.678, Accuracy: 0.786
Epoch 17/200, Loss: 0.673, Accuracy: 0.714
Epoch 18/200, Loss: 0.669, Accuracy: 0.714
Epoch 19/200, Loss: 0.665, Accuracy: 0.714
Epoch 20/200, Loss: 0.661, Accuracy: 0.714
Epoch 21/200, Loss: 0.657, Accuracy: 0.714
Epoch 22/200, Loss: 0.653, Accuracy: 0.786
Epoch 23/200, Loss: 0.647, Accuracy: 0.786
Epoch 24/200, Loss: 

Epoch 191/200, Loss: 0.433, Accuracy: 0.857
Epoch 192/200, Loss: 0.433, Accuracy: 0.857
Epoch 193/200, Loss: 0.432, Accuracy: 0.857
Epoch 194/200, Loss: 0.432, Accuracy: 0.857
Epoch 195/200, Loss: 0.432, Accuracy: 0.857
Epoch 196/200, Loss: 0.432, Accuracy: 0.857
Epoch 197/200, Loss: 0.431, Accuracy: 0.857
Epoch 198/200, Loss: 0.430, Accuracy: 0.857
Epoch 199/200, Loss: 0.430, Accuracy: 0.857
Epoch 200/200, Loss: 0.430, Accuracy: 0.857


In [57]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for inputs, labels in test_dataloader:

        inputs = inputs.float()
        labels = labels.float().view(-1,1)

        outputs = model(inputs)
        
        preds = (outputs > 0.5).float()

        all_preds.append(preds)
        all_labels.append(labels)

preds = torch.cat(all_preds)
labels = torch.cat(all_labels)

y_pred = preds.numpy()
y_true = labels.numpy()

y_pred = y_pred.flatten()
y_true = y_true.flatten()

from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

cm = confusion_matrix(y_true, y_pred)

print(cm)

accuracy = accuracy_score(y_true, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)


[[ 7 24]
 [ 5 14]]
Accuracy: 0.42
Precision: 0.3684210526315789
Recall: 0.7368421052631579
F1 Score: 0.49122807017543857


In [55]:
for param in model.parameters():
    print(param)

Parameter containing:
tensor([[ 0.0680,  0.5513,  0.0086,  0.2240, -0.0640,  0.4669, -0.0963],
        [ 0.1975,  0.3860, -0.1887, -0.0893, -0.1699,  0.0137, -0.4294],
        [-0.2353, -0.5884, -0.1637,  0.3971, -0.5339, -0.1635, -0.0588],
        [-0.3148, -0.0031,  0.0646, -0.6289, -0.2238, -0.6847, -0.2133],
        [-0.2486, -0.2281,  0.2538, -0.1702,  0.0901, -0.1838, -1.0656]],
       requires_grad=True)
Parameter containing:
tensor([0.5123, 0.0742, 0.1892, 0.7849, 0.1224], requires_grad=True)
Parameter containing:
tensor([[ 0.6104,  0.8807,  0.0569,  0.3409, -0.4660],
        [-0.2201, -0.4362,  0.5958,  0.2953,  0.7580],
        [ 0.4132,  0.5427, -0.4092,  0.4052, -0.5903],
        [-0.0895, -0.3071,  0.1091,  0.3950,  0.2527]], requires_grad=True)
Parameter containing:
tensor([-0.1544,  0.5753,  0.1979,  0.3278], requires_grad=True)
Parameter containing:
tensor([[-0.6624,  0.9554, -0.4510,  0.7819],
        [ 0.6109, -0.5479,  0.5607, -0.4492],
        [-0.2364, -0.1843, -0.